# yeast-0-5-6-7-9_vs_4: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('35_yeast-0-5-6-7-9_vs_4_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,49.287,38.259,91.000,25.636,99.583,62.610,64.659,88.105,43.773
smote,74.108,53.411,52.021,60.909,93.200,77.054,59.993,87.905,63.760
bsmote,71.494,52.907,53.753,55.909,94.660,75.285,59.743,88.481,62.201
ros,73.448,51.573,47.430,59.909,92.670,76.290,60.402,89.174,62.511
adasyn,72.633,48.847,45.134,59.909,91.208,75.559,57.994,87.584,60.740
casb,73.397,54.933,53.845,58.909,94.447,76.678,61.156,89.085,64.165
casb_relaxed,72.608,52.231,51.457,58.909,93.616,76.263,58.446,88.516,62.420
casb_edited,75.605,51.718,44.880,63.909,91.305,77.607,59.357,88.364,63.662
casb_ros,74.266,56.006,55.062,59.818,94.447,77.133,62.476,89.009,65.136


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,52.209,38.664,71.048,29.818,98.325,64.071,52.468,81.516,45.436
smote,75.625,48.308,40.431,65.818,87.859,76.838,45.713,83.085,61.966
bsmote,77.659,49.999,40.452,68.727,88.370,78.548,48.038,84.365,63.829
ros,74.828,45.684,36.478,64.909,87.120,76.014,36.692,79.505,60.256
adasyn,75.667,45.081,35.232,67.818,85.129,76.474,43.834,83.275,60.374
casb,76.801,51.712,44.193,65.909,90.252,78.081,48.273,81.104,64.256
casb_relaxed,75.483,46.825,37.568,65.818,87.322,76.570,48.893,84.517,61.154
casb_edited,75.580,44.129,33.198,67.727,84.910,76.319,47.097,84.462,59.854
casb_ros,74.689,48.655,41.129,62.909,89.524,76.217,45.295,80.715,61.672


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,50.011,36.815,78.917,27.636,98.746,63.191,59.517,89.610,43.413
smote,74.747,56.606,59.412,60.818,94.241,77.530,58.359,89.691,65.676
bsmote,72.077,55.797,60.894,55.818,95.502,75.660,57.768,90.211,63.937
ros,69.831,53.908,58.635,52.273,96.020,74.146,57.688,90.676,61.870
adasyn,70.691,52.870,56.977,54.182,94.868,74.525,57.542,89.648,61.781
casb,73.374,58.647,62.892,57.000,96.336,76.668,58.902,90.141,66.010
casb_relaxed,73.524,56.656,58.652,57.909,95.186,76.548,58.559,91.044,65.090
casb_edited,75.753,55.345,50.617,62.818,93.293,78.055,56.582,90.789,65.549
casb_ros,69.156,53.642,60.338,51.182,96.230,73.706,57.016,89.703,61.399


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,55.412,44.034,82.429,31.636,98.956,65.296,63.052,85.187,49.723
smote,77.932,53.529,45.205,68.364,90.467,79.415,61.540,84.706,65.730
bsmote,78.508,55.433,47.578,68.455,91.725,80.090,62.433,85.055,66.971
ros,76.918,49.281,39.060,68.364,88.577,78.470,58.883,84.820,63.100
adasyn,77.583,50.390,40.556,69.364,88.368,78.866,61.693,84.449,63.986
casb,79.061,56.553,48.906,69.364,91.831,80.597,63.360,85.163,67.807
casb_relaxed,77.821,53.150,44.431,68.364,90.678,79.521,62.138,84.926,65.486
casb_edited,77.533,47.968,36.943,70.364,86.797,78.580,60.454,84.893,62.751
casb_ros,78.548,54.155,45.574,69.364,90.677,80.020,62.937,85.240,66.352


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,51.730,39.443,80.848,28.682,98.902,63.792,59.924,86.104,45.586
smote,75.603,52.964,49.267,63.977,91.442,77.709,56.401,86.347,64.283
bsmote,74.935,53.534,50.669,62.227,92.564,77.396,56.995,87.028,64.234
ros,73.757,50.112,45.401,61.364,91.096,76.230,53.416,86.044,61.934
adasyn,74.144,49.297,44.475,62.818,89.894,76.356,55.266,86.239,61.720
casb,75.658,55.461,52.459,62.795,93.217,78.006,57.923,86.373,65.560
casb_relaxed,74.859,52.216,48.027,62.750,91.701,77.225,57.009,87.251,63.537
casb_edited,76.118,49.790,41.410,66.205,89.076,77.640,55.873,87.127,62.954
casb_ros,74.165,53.114,50.526,60.818,92.720,76.769,56.931,86.167,63.640


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,75.605,smote,74.108,1.497,WIN
1,f1,casb_ros,56.006,smote,53.411,2.595,WIN
2,precision,casb_ros,55.062,bsmote,53.753,1.310,WIN
3,recall,casb_edited,63.909,smote,60.909,3.000,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,76.801,bsmote,77.659,-0.858,LOSS
1,f1,casb,51.712,bsmote,49.999,1.712,WIN
2,precision,casb,44.193,bsmote,40.452,3.741,WIN
3,recall,casb_edited,67.727,bsmote,68.727,-1.000,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,75.753,smote,74.747,1.007,WIN
1,f1,casb,58.647,smote,56.606,2.041,WIN
2,precision,casb,62.892,bsmote,60.894,1.998,WIN
3,recall,casb_edited,62.818,smote,60.818,2.000,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,76.118,smote,75.603,0.515,WIN
1,f1,casb,55.461,bsmote,53.534,1.927,WIN
2,precision,casb,52.459,bsmote,50.669,1.790,WIN
3,recall,casb_edited,66.205,smote,63.977,2.227,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.426401,0.307692,1.000000,0.181818,1.000000,0.590909,0.649638,0.921531,0.367047,None
1,42,1,svm,smote,0.50,0.726794,0.600000,0.666667,0.545455,0.968421,0.756938,0.632430,0.932057,0.663397,None
2,42,1,svm,bsmote,0.75,0.722833,0.571429,0.600000,0.545455,0.957895,0.751675,0.586113,0.921531,0.647131,None
3,42,1,svm,ros,0.75,0.718851,0.545455,0.545455,0.545455,0.947368,0.746411,0.558491,0.911005,0.632153,None
4,42,1,svm,adasyn,0.50,0.722833,0.571429,0.600000,0.545455,0.957895,0.751675,0.580076,0.911005,0.647131,None
5,42,1,svm,casb,0.50,0.726794,0.600000,0.666667,0.545455,0.968421,0.756938,0.592495,0.920574,0.663397,None
6,42,1,svm,casb_relaxed,0.75,0.722833,0.571429,0.600000,0.545455,0.957895,0.751675,0.605640,0.926316,0.647131,None
7,42,1,svm,casb_edited,0.50,0.722833,0.571429,0.600000,0.545455,0.957895,0.751675,0.585635,0.914833,0.647131,None
8,42,1,svm,casb_ros,0.75,0.780749,0.636364,0.636364,0.636364,0.957895,0.797129,0.655409,0.934928,0.708556,None
9,42,1,knn,original,0.50,0.301511,0.166667,1.000000,0.090909,1.000000,0.545455,0.616702,0.864115,0.234089,None
